## Robustness to corruptions — Wake Vision

This notebook evaluates models trained on **Wake Vision** (person vs non-person) using the **19 corruptions** from Hendrycks & Dietterich (2019), applied **on-the-fly** via `imagecorruptions`.

- **15 common** (used in the official mCE): gaussian_noise, shot_noise, impulse_noise, defocus_blur, glass_blur, motion_blur, zoom_blur, snow, frost, fog, brightness, contrast, elastic_transform, pixelate, jpeg_compression
- **4 extra/validation** (not part of the official mCE): speckle_noise, gaussian_blur, saturate, spatter

### Why on-the-fly?

On CIFAR-10/100, corruption datasets (CIFAR-10-C, CIFAR-100-C) are shipped as pre-generated `.npy` because images are 32×32. 
On Wake Vision, the test set has ~55K images at varied sizes (resized to 224×224). Pre-generating and saving all corruptions to disk would use **~780 GB** (19 corruptions × 5 severities × 55K × 224×224×3). 

The solution is to use `imagecorruptions` to apply each corruption per image, evaluate, and discard—without writing anything to disk.

### Metrics
- **CE (Corruption Error):** `Σ_s err_model(s) / Σ_s err_baseline(s)` (severities 1–5)
- **mCE:** mean CE over the **15 common corruptions** (official metric, Hendrycks 2019)
- **mCE-19:** mean CE over all 19 corruptions (informational)
- **Relative mCE:** adjusts for each model’s clean accuracy

In [ ]:
from pathlib import Path


def project_root() -> Path:
    """Resolve repo root (repo root, notebooks/, or other subfolders)."""
    cwd = Path.cwd().resolve()
    for d in [cwd, *cwd.parents]:
        km = d / "keras_models"
        if km.is_dir() and any(km.glob("*.keras")):
            return d
        if (d / ".git").exists() and (d / "results").is_dir():
            return d
    if (cwd / "datasets").is_dir() or (cwd / "results").is_dir():
        return cwd
    if (cwd.parent / "datasets").is_dir() or (cwd.parent / "results").is_dir():
        return cwd.parent
    return cwd


ROOT = project_root()


In [ ]:
from __future__ import annotations

import gc
import json
import os
from pathlib import Path

# Set TF_GPU_ALLOCATOR to "cuda_malloc_async" for better GPU memory management (optional)
os.environ.setdefault("TF_GPU_ALLOCATOR", "cuda_malloc_async")

import numpy as np
import tensorflow as tf

SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 128
CLEAN_BATCH_SIZE = 16
CORRUPTION_BATCH_SIZE = 16
PREFETCH_SIZE = 1
NUM_CLASSES = 2
GPU_ID = 1

tf.keras.utils.set_random_seed(SEED)
class_names = ["no_person", "person"]

print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("TF_GPU_ALLOCATOR:", os.environ.get("TF_GPU_ALLOCATOR"))

2026-03-20 21:53:43.292664: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-20 21:53:43.321471: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-20 21:53:44.003849: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow: 2.20.0
NumPy: 2.4.2
TF_GPU_ALLOCATOR: cuda_malloc_async


In [ ]:
# GPU
gpus = tf.config.list_physical_devices("GPU")
print("Available GPUs:", gpus)

if gpus:
    if GPU_ID is not None:
        chosen = gpus[GPU_ID]
        tf.config.set_visible_devices([chosen], "GPU")
        tf.config.experimental.set_memory_growth(chosen, True)
        print(f"✓ Using only GPU {GPU_ID}: {chosen.name}")
    else:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✓ Using all the {len(gpus)} GPUs")
else:
    print("⚠ No GPU detected")

Available GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
✓ Usando apenas GPU 1: /physical_device:GPU:1


## 1 — Load .keras models

In [ ]:
from tensorflow.keras import layers

# Register custom layer (required to load MCUNet)
@tf.keras.utils.register_keras_serializable(package="MCUNet")
class ImageNetNormalization(layers.Layer):
    """ImageNet normalization: (x/255 - mean) / std"""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.mean = tf.constant([0.485, 0.456, 0.406], dtype=tf.float32)
        self.std = tf.constant([0.229, 0.224, 0.225], dtype=tf.float32)

    def call(self, x):
        return (x / 255.0 - self.mean) / self.std

    def get_config(self):
        return super().get_config()


MODELS = [
    ("MCUNet", ROOT / "keras_models" / "MCUNet_WakeVision.keras"),
    ("MobileNetV3Small", ROOT / "keras_models" / "mobilenetv3_small_wakevision.keras"),
    ("EfficientNetB0", ROOT / "keras_models" / "EfficientNetB0_WakeVision.keras"),
]


def collect_model_specs(specs):
    out = []
    for name, path in specs:
        if not path.exists():
            print(f"WARNING: {path} not found, skipping...")
            continue
        out.append((name, path, None))
        print(f"OK: {name} <- {path}")
    return out


def load_frozen_model(path: Path) -> tf.keras.Model:
    model = tf.keras.models.load_model(path)
    model.trainable = False
    return model


def release_model_memory(model: tf.keras.Model | None = None) -> None:
    if model is not None:
        del model
    tf.keras.backend.clear_session()
    gc.collect()


models = collect_model_specs(MODELS)

if not models:
    raise RuntimeError("Nenhum model encontrado!")

print(f"\n{len(models)} models available")

OK: MobileNetV3Small <- keras_models/mobilenetv3_small_wakevision.keras
OK: EfficientNetB0 <- keras_models/EfficientNetB0_WakeVision.keras
OK: MCUNet <- keras_models/MCUNet_WakeVision.keras

3 models available


## 2 — Load Wake Vision test set

In [ ]:
import tensorflow_datasets as tfds

DATA_DIR = "/anonymous/anonymous/anonymous/data"

ds_test = tfds.load(
    "wake_vision",
    split="test",
    data_dir=DATA_DIR,
    as_supervised=False,
    shuffle_files=False,
)

# Filter examples with person == -1 (benchmark "far distance")
ds_test = ds_test.filter(lambda x: x["person"] >= 0)


def _to_supervised(example):
    return example["image"], tf.cast(example["person"], tf.int64)


ds_test = ds_test.map(_to_supervised, num_parallel_calls=tf.data.AUTOTUNE)

# Materialize the test set in memory (needed for per-image corruptions)
print("Materializing test set in memory (may take a while)...")
images_list = []
labels_list = []
for img, lab in ds_test:
    images_list.append(img.numpy())
    labels_list.append(lab.numpy())

y_test = np.array(labels_list, dtype="int64")
print(f"Test set: {len(images_list)} images")
print(f"Labels: {y_test.shape}, classes: {np.unique(y_test)}")
print(f"Distribution: no_person={np.sum(y_test == 0)}, person={np.sum(y_test == 1)}")

I0000 00:00:1774054441.858878   15847 gpu_process_state.cc:208] Using CUDA malloc Async allocator for GPU: 1
I0000 00:00:1774054441.859649   15847 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22138 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:02:00.0, compute capability: 8.9
2026-03-20 21:54:01.987025: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:396] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608


Materializing test set in memory (may take a while)...
Test set: 55762 imagens
Labels: (55762,), classes: [0 1]
Distribution: no_person=27881, person=27881


2026-03-20 21:54:16.067203: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


## 3 — Evaluation with on-the-fly corruptions

In [ ]:
import inspect
from concurrent.futures import ThreadPoolExecutor

import imagecorruptions as ic_module
from imagecorruptions import corrupt, get_corruption_names
import imagecorruptions.corruptions as ic_corruptions
from skimage import filters as sk_filters


# ── Patch 1: compatibility with new scikit-image (removed `multichannel` arg) ──
def _patch_gaussian_multichannel_compat() -> None:
    current = ic_corruptions.gaussian
    if "multichannel" in inspect.signature(current).parameters:
        return  # old version, no patch needed

    def _gaussian_compat(image, sigma=1, multichannel=None, **kwargs):
        if multichannel is not None and "channel_axis" not in kwargs:
            kwargs["channel_axis"] = -1 if multichannel else None
        return current(image, sigma=sigma, **kwargs)

    ic_corruptions.gaussian = _gaussian_compat
    sk_filters.gaussian = _gaussian_compat


# ── Patch 2: vectorized glass_blur (numpy) ──────────────────────────────────────
# The original uses two H×W Python for-loops per image.
# For 224×224 × 55K images that is ~5 billion Python iters—hours of runtime.
# This version replaces the loops with equivalent numpy operations (~100–1000x faster).
def _fast_glass_blur(x, severity: int = 1):
    c = [(0.7, 1, 2), (0.9, 2, 1), (1, 2, 3), (1.1, 3, 2), (1.5, 4, 2)][severity - 1]
    sigma, max_delta, iters = c[0], c[1], c[2]

    # gaussian: channel_axis=-1 to avoid depending on patch #1
    arr = np.uint8(
        np.clip(
            sk_filters.gaussian.__wrapped__(np.array(x) / 255.0, sigma=sigma, channel_axis=-1)
            if hasattr(sk_filters.gaussian, "__wrapped__")
            else sk_filters.gaussian(np.array(x) / 255.0, sigma=sigma, channel_axis=-1),
            0, 1,
        ) * 255
    )

    h, w = arr.shape[0], arr.shape[1]
    hs = np.arange(max_delta, h - max_delta)
    ws = np.arange(max_delta, w - max_delta)
    yy, xx = np.meshgrid(hs, ws, indexing="ij")  # shape (H', W')

    for _ in range(iters):
        dy = np.random.randint(-max_delta, max_delta + 1, size=yy.shape)
        dx = np.random.randint(-max_delta, max_delta + 1, size=xx.shape)
        yp = np.clip(yy + dy, 0, h - 1)
        xp = np.clip(xx + dx, 0, w - 1)
        buf = arr.copy()
        arr[yy, xx] = buf[yp, xp]
        arr[yp, xp] = buf[yy, xx]

    result = np.clip(
        sk_filters.gaussian.__wrapped__(arr / 255.0, sigma=sigma, channel_axis=-1)
        if hasattr(sk_filters.gaussian, "__wrapped__")
        else sk_filters.gaussian(arr / 255.0, sigma=sigma, channel_axis=-1),
        0, 1,
    ) * 255
    return result


def _apply_patches() -> None:
    _patch_gaussian_multichannel_compat()

    # Patch module attribute (for direct call paths)
    ic_corruptions.glass_blur = _fast_glass_blur

    # CRITICAL: corrupt() uses corruption_dict built at import with
    # direct function references — those must be updated as well.
    if hasattr(ic_module, "corruption_dict") and "glass_blur" in ic_module.corruption_dict:
        ic_module.corruption_dict["glass_blur"] = _fast_glass_blur
        print("  corruption_dict['glass_blur'] updated")
    else:
        print("  WARNING: corruption_dict not found — corrupt() may use a slow version")

    print("Patches applied: gaussian(multichannel) + vectorized glass_blur")


_apply_patches()

COMMON_CORRUPTIONS = get_corruption_names("common")  # 15 for default mCE
ALL_CORRUPTIONS = get_corruption_names("all")         # 19 total (15 common + 4 extra)
EXTRA_CORRUPTIONS = [c for c in ALL_CORRUPTIONS if c not in COMMON_CORRUPTIONS]

# We use all 19 corruptions in evaluation
CORRUPTIONS = ALL_CORRUPTIONS
SEVERITIES = [1, 2, 3, 4, 5]
AUTOTUNE = tf.data.AUTOTUNE

# CPU parallelization for all corruptions per batch.
CORRUPTION_NUM_WORKERS = max(1, min(8, os.cpu_count() or 1))
_CORRUPTION_EXECUTOR = ThreadPoolExecutor(max_workers=CORRUPTION_NUM_WORKERS)

print(f"Common corruptions ({len(COMMON_CORRUPTIONS)}):")
for c in COMMON_CORRUPTIONS:
    print(f"  - {c}")
print(f"\nExtra/validation corruptions ({len(EXTRA_CORRUPTIONS)}):")
for c in EXTRA_CORRUPTIONS:
    print(f"  - {c}")
print(f"\nTotal: {len(CORRUPTIONS)} corruptions")
print(f"Corruption workers (CPU): {CORRUPTION_NUM_WORKERS}")


def preprocess_for_model(image: np.ndarray, label: int):
    """Resize + cast to float32. No normalization here—each model does it internally."""
    image = tf.cast(image, tf.float32)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE), method="bilinear")
    return image, label


def make_ds_from_arrays(x: np.ndarray, y: np.ndarray, batch_size: int = BATCH_SIZE):
    """Creates tf.data.Dataset from numpy arrays."""
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    ds = ds.map(
        lambda img, lab: (tf.cast(tf.image.resize(tf.cast(img, tf.float32), (IMG_SIZE, IMG_SIZE)), tf.float32), lab),
        num_parallel_calls=AUTOTUNE,
    )
    ds = ds.batch(batch_size).prefetch(PREFETCH_SIZE)
    return ds


def accuracy_on_ds(model: tf.keras.Model, ds: tf.data.Dataset) -> float:
    correct = total = 0
    for xb, yb in ds:
        pred = tf.argmax(model(xb, training=False), axis=-1, output_type=yb.dtype)
        correct += int(tf.reduce_sum(tf.cast(pred == yb, tf.int32)).numpy())
        total += int(yb.shape[0])
        del pred
    return correct / max(total, 1)


def _corrupt_and_resize_one(args):
    img, _, corruption_name, severity = args

    if img.dtype != np.uint8:
        img = np.clip(img, 0, 255).astype(np.uint8)

    # Always use corrupt() wrapper for compatibility with corruptions
    # that expect PIL internally (e.g. pixelate and jpeg_compression).
    corrupted = corrupt(img, corruption_name=corruption_name, severity=severity)

    resized = tf.image.resize(tf.cast(corrupted, tf.float32), (IMG_SIZE, IMG_SIZE))
    return resized.numpy()


def accuracy_on_corruption_batches(
    model: tf.keras.Model,
    images_list: list[np.ndarray],
    labels: np.ndarray,
    corruption: str,
    severity: int,
    batch_size: int = CORRUPTION_BATCH_SIZE,
) -> float:
    correct = total = 0

    for start in range(0, len(images_list), batch_size):
        end = min(start + batch_size, len(images_list))
        batch_images = images_list[start:end]
        batch_labels = labels[start:end]

        args_iter = [(img, None, corruption, severity) for img in batch_images]
        if CORRUPTION_NUM_WORKERS > 1 and len(batch_images) > 1:
            resized_batch = list(_CORRUPTION_EXECUTOR.map(_corrupt_and_resize_one, args_iter))
        else:
            resized_batch = [_corrupt_and_resize_one(args) for args in args_iter]

        xb = tf.convert_to_tensor(np.stack(resized_batch, axis=0), dtype=tf.float32)
        yb = tf.convert_to_tensor(batch_labels, dtype=tf.int64)
        pred = tf.argmax(model(xb, training=False), axis=-1, output_type=yb.dtype)

        correct += int(tf.reduce_sum(tf.cast(pred == yb, tf.int32)).numpy())
        total += end - start

        del resized_batch, xb, yb, pred

    return correct / max(total, 1)


def eval_corruption(
    model: tf.keras.Model,
    images_list: list[np.ndarray],
    y: np.ndarray,
    corruption: str,
    severities: list[int] = SEVERITIES,
) -> dict:
    """Evaluate one model on one corruption across all severities."""
    out = {"corruption": corruption}

    for s in severities:
        print(f"    severity={s} ...", end=" ", flush=True)
        acc = accuracy_on_corruption_batches(
            model=model,
            images_list=images_list,
            labels=y,
            corruption=corruption,
            severity=s,
            batch_size=CORRUPTION_BATCH_SIZE,
        )
        out[f"acc_s{s}"] = float(acc)
        print(f"acc={acc:.4f}")

    out["acc_mean_1_5"] = float(np.mean([out[f"acc_s{s}"] for s in severities]))
    return out


def err_from_acc(acc: float) -> float:
    return 1.0 - float(acc)


def ce_from_accs(acc_model: dict, acc_base: dict, severities=SEVERITIES) -> float:
    """CE = Σ_s err_model(s) / Σ_s err_baseline(s)"""
    num = sum(err_from_acc(acc_model[f"acc_s{s}"]) for s in severities)
    den = sum(err_from_acc(acc_base[f"acc_s{s}"]) for s in severities)
    return num / den if den > 0 else float("nan")

  corruption_dict['glass_blur'] atualizado
Patches aplicados: gaussian(multichannel) + glass_blur vetorizado
Common corruptions (15):
  - gaussian_noise
  - shot_noise
  - impulse_noise
  - defocus_blur
  - glass_blur
  - motion_blur
  - zoom_blur
  - snow
  - frost
  - fog
  - brightness
  - contrast
  - elastic_transform
  - pixelate
  - jpeg_compression

Extra/validation corruptions (4):
  - speckle_noise
  - gaussian_blur
  - spatter
  - saturate

Total: 19 corruptions
Workers de corruption (CPU): 8


In [ ]:
# NumPy 2.x compatibility for legacy libs (imagecorruptions)
if not hasattr(np, "float_"):
    np.float_ = np.float64
if not hasattr(np, "complex_"):
    np.complex_ = np.complex128

print("NumPy 2 compatibility applied: np.float_ -> np.float64, np.complex_ -> np.complex128")

Compat NumPy 2 aplicado: np.float_ -> np.float64, np.complex_ -> np.complex128


## 4 — Clean accuracy + baseline

In [ ]:
# Baseline = lowest accuracy
    print(f"Evaluating clean set with batch_size={CLEAN_BATCH_SIZE} ...")

def clean_generator():
    for img, lab in zip(images_list, y_test):
        yield img, lab

clean_signature = (
    tf.TensorSpec(shape=(None, None, 3), dtype=tf.uint8),
    tf.TensorSpec(shape=(), dtype=tf.int64),
)

clean_ds = tf.data.Dataset.from_generator(
    clean_generator,
    output_signature=clean_signature,
)
clean_ds = clean_ds.map(preprocess_for_model, num_parallel_calls=AUTOTUNE)
clean_ds = clean_ds.batch(CLEAN_BATCH_SIZE).prefetch(PREFETCH_SIZE)

clean_accs = {}
clean_acc_baseline = None

for name, path, _ in models:
    print(f"- {name}")
    model = load_frozen_model(path)
    try:
        acc = accuracy_on_ds(model, clean_ds)
        clean_accs[name] = acc
        print(f"  acc_clean = {acc:.4f}")
    finally:
        release_model_memory(model)

# Pick the lowest-accuracy model as the baseline automatically
BASELINE_NAME = min(clean_accs, key=clean_accs.get)
clean_acc_baseline = clean_accs[BASELINE_NAME]

print("\nClean summary:")
for name, acc in clean_accs.items():
    print(f"  {name:<20} {acc:.4f}")

print(f"\nBaseline for CE/mCE: {BASELINE_NAME} (acc_clean={clean_acc_baseline:.4f})")

Avaliando clean set com batch_size=16 ...
- MobileNetV3Small
  acc_clean = 0.8613
- EfficientNetB0
  acc_clean = 0.8773
- MCUNet
  acc_clean = 0.8127

Resumo clean:
  MobileNetV3Small     0.8613
  EfficientNetB0       0.8773
  MCUNet               0.8127

Baseline for CE/mCE: MCUNet (acc_clean=0.8127)


## 5 — Corruption evaluation (with JSONL persistence)

Each result is written line by line. If the kernel restarts, already-finished corruptions are skipped.

In [ ]:
# JSONL persistence
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if 'clean_accs' in globals() and clean_accs:
    BASELINE_NAME = min(clean_accs, key=clean_accs.get)
elif 'BASELINE_NAME' not in globals():
    BASELINE_NAME = models[0][0]

RUN_TAG = f"wakevision_baseline-{BASELINE_NAME}".replace(" ", "_")
BASELINE_ACC_PATH = RESULTS_DIR / f"{RUN_TAG}_baseline_acc.json"
CE_JSONL_PATH = RESULTS_DIR / f"{RUN_TAG}_ce_results.jsonl"

FORCE_RECOMPUTE = False


def _load_json(path: Path, default):
    if not path.exists():
        return default
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return default


def _atomic_write_json(path: Path, obj) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    os.replace(tmp, path)


def _load_jsonl(path: Path):
    rows = []
    if not path.exists():
        return rows
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line:
            try:
                rows.append(json.loads(line))
            except Exception:
                pass
    return rows


def _append_jsonl(path: Path, row: dict) -> None:
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
        f.flush()
        os.fsync(f.fileno())


existing = _load_jsonl(CE_JSONL_PATH)
baseline_acc = _load_json(BASELINE_ACC_PATH, {})

print(f"Avaliando {len(models)} models x {len(CORRUPTIONS)} corruptions x {len(SEVERITIES)} severities")
print(f"Corruption batch size: {CORRUPTION_BATCH_SIZE}")
print("Each model is loaded separately to limit VRAM use\n")

results = {name: {} for name, _, _ in models}

for row in existing:
    model_name = row.get("model")
    corruption = row.get("corruption")
    if model_name not in results or not corruption:
        continue
    results[model_name].setdefault(corruption, {})
    for severity in SEVERITIES:
        key = f"acc_s{severity}"
        if key in row:
            results[model_name][corruption][key] = row[key]
    if "acc_mean_1_5" in row:
        results[model_name][corruption]["acc_mean_1_5"] = row["acc_mean_1_5"]

for name, path, _ in models:
    print(f"\n{'=' * 60}")
    print(f"Model: {name}")
    model = load_frozen_model(path)

    try:
        for c_idx, corruption in enumerate(CORRUPTIONS):
            print(f"[{c_idx + 1}/{len(CORRUPTIONS)}] {corruption}")
            results[name].setdefault(corruption, {})

            cached_all = (
                (not FORCE_RECOMPUTE)
                and all(f"acc_s{severity}" in results[name][corruption] for severity in SEVERITIES)
            )
            if cached_all:
                print("  cached")
                continue

            for severity in SEVERITIES:
                key = f"acc_s{severity}"
                if not FORCE_RECOMPUTE and key in results[name][corruption]:
                    print(f"  severity={severity}: cached acc={results[name][corruption][key]:.4f}")
                    continue

                print(f"  severity={severity}: evaluating...", end=" ", flush=True)
                acc = accuracy_on_corruption_batches(
                    model=model,
                    images_list=images_list,
                    labels=y_test,
                    corruption=corruption,
                    severity=severity,
                    batch_size=CORRUPTION_BATCH_SIZE,
                )
                results[name][corruption][key] = float(acc)
                print(f"acc={acc:.4f}")

            accs = results[name][corruption]
            if all(f"acc_s{severity}" in accs for severity in SEVERITIES):
                acc_mean = float(np.mean([accs[f"acc_s{severity}"] for severity in SEVERITIES]))
                accs["acc_mean_1_5"] = acc_mean

                row = {
                    "model": name,
                    "corruption": corruption,
                    "acc_mean_1_5": acc_mean,
                }
                for severity in SEVERITIES:
                    row[f"acc_s{severity}"] = accs[f"acc_s{severity}"]

                if name == BASELINE_NAME:
                    row["ce_mean_1_5"] = 1.0
                    for severity in SEVERITIES:
                        row[f"ce_s{severity}"] = 1.0

                    baseline_acc[corruption] = {
                        f"acc_s{severity}": accs[f"acc_s{severity}"] for severity in SEVERITIES
                    }
                    baseline_acc[corruption]["acc_mean_1_5"] = acc_mean
                    _atomic_write_json(BASELINE_ACC_PATH, baseline_acc)
                else:
                    base_accs = baseline_acc.get(corruption, {})
                    if all(f"acc_s{severity}" in base_accs for severity in SEVERITIES):
                        num_err = sum(err_from_acc(accs[f"acc_s{severity}"]) for severity in SEVERITIES)
                        den_err = sum(err_from_acc(base_accs[f"acc_s{severity}"]) for severity in SEVERITIES)
                        row["ce_mean_1_5"] = num_err / den_err if den_err > 0 else float("nan")
                        for severity in SEVERITIES:
                            base_err = err_from_acc(base_accs[f"acc_s{severity}"])
                            row[f"ce_s{severity}"] = (
                                err_from_acc(accs[f"acc_s{severity}"]) / base_err if base_err > 0 else float("nan")
                            )

                existing = [
                    r for r in existing
                    if not (r.get("model") == name and r.get("corruption") == corruption)
                ]
                existing.append(row)

                with CE_JSONL_PATH.open("w", encoding="utf-8") as f:
                    for saved_row in existing:
                        f.write(json.dumps(saved_row, ensure_ascii=False) + "\n")
                    f.flush()
                    os.fsync(f.fileno())

                ce_value = row.get("ce_mean_1_5")
                ce_display = f"{ce_value:.4f}" if isinstance(ce_value, (int, float, np.floating)) else "-"
                print(f"  saved: acc_mean={acc_mean:.4f}  CE={ce_display}")
                
                gc.collect()  # manual collect between corruptions to free memory
    finally:
        release_model_memory(model)

print(f"\n{'='*60}")
print("Evaluation complete!")
print(f"  Results: {CE_JSONL_PATH}")
print(f"  Baseline:   {BASELINE_ACC_PATH}")

Avaliando 3 models x 19 corruptions x 5 severidades
Corruption batch size: 16
Each model is loaded separately to limit VRAM use


Model: MobileNetV3Small
[1/19] gaussian_noise
  cached
[2/19] shot_noise
  cached
[3/19] impulse_noise
  cached
[4/19] defocus_blur
  cached
[5/19] glass_blur
  cached
[6/19] motion_blur
  cached
[7/19] zoom_blur
  cached
[8/19] snow
  cached
[9/19] frost
  cached
[10/19] fog
  cached
[11/19] brightness
  cached
[12/19] contrast
  cached
[13/19] elastic_transform
  cached
[14/19] pixelate
  cached
[15/19] jpeg_compression
  cached
[16/19] speckle_noise
  cached
[17/19] gaussian_blur
  cached
[18/19] spatter
  cached
[19/19] saturate
  cached

Model: EfficientNetB0
[1/19] gaussian_noise
  cached
[2/19] shot_noise
  cached
[3/19] impulse_noise
  cached
[4/19] defocus_blur
  cached
[5/19] glass_blur
  cached
[6/19] motion_blur
  cached
[7/19] zoom_blur
  cached
[8/19] snow
  cached
[9/19] frost
  cached
[10/19] fog
  cached
[11/19] brightness
  cached
[12/19] c

In [ ]:
# === Recalculating CE using MCUNet as baseline ===
NEW_BASELINE_NAME = "MCUNet"
OLD_JSONL_PATH = CE_JSONL_PATH

RUN_TAG = f"wakevision_baseline-{NEW_BASELINE_NAME}".replace(" ", "_")
BASELINE_ACC_PATH = RESULTS_DIR / f"{RUN_TAG}_baseline_acc.json"
CE_JSONL_PATH = RESULTS_DIR / f"{RUN_TAG}_ce_results.jsonl"

old_results = _load_jsonl(OLD_JSONL_PATH)

# 1. Isolar dados do novo baseline
new_baseline_accs = {}
for row in old_results:
    if row.get("model") == NEW_BASELINE_NAME:
        c = row["corruption"]
        new_baseline_accs[c] = {k: v for k, v in row.items() if k.startswith("acc_")}

# Salva baseline_acc.json novo
_atomic_write_json(BASELINE_ACC_PATH, new_baseline_accs)

# 2. Recompute CE for all models
new_results = []
for row in old_results:
    new_row = row.copy()
    c = row["corruption"]
    b_accs = new_baseline_accs.get(c, {})
    
    if c in new_baseline_accs and all(f"acc_s{s}" in row for s in SEVERITIES):
        if row["model"] == NEW_BASELINE_NAME:
            new_row["ce_mean_1_5"] = 1.0
            for s in SEVERITIES:
                new_row[f"ce_s{s}"] = 1.0
        else:
            num_err = sum(err_from_acc(row[f"acc_s{s}"]) for s in SEVERITIES)
            den_err = sum(err_from_acc(b_accs[f"acc_s{s}"]) for s in SEVERITIES)
            new_row["ce_mean_1_5"] = num_err / den_err if den_err > 0 else float("nan")
            
            for s in SEVERITIES:
                base_err = err_from_acc(b_accs[f"acc_s{s}"])
                model_err = err_from_acc(row[f"acc_s{s}"])
                new_row[f"ce_s{s}"] = model_err / base_err if base_err > 0 else float("nan")
    new_results.append(new_row)

# 3. Gravar novo arquivo JSONL
with CE_JSONL_PATH.open("w", encoding="utf-8") as f:
    for r in new_results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

BASELINE_NAME = NEW_BASELINE_NAME
print(f"Recalculou todos os CEs usando {BASELINE_NAME} como baseline!")
print(f"Novo jsonl salvo em: {CE_JSONL_PATH}")

Recalculou todos os CEs usando MCUNet como baseline!
Novo jsonl salvo em: results/wakevision_baseline-MCUNet_ce_results.jsonl


## 6 — Resumo mCE

In [ ]:
# Load results (works after kernel restarts too)
if 'clean_accs' in globals() and clean_accs:
    BASELINE_NAME = min(clean_accs, key=clean_accs.get)
elif 'BASELINE_NAME' not in globals():
    BASELINE_NAME = "MCUNet"  # auto fallback (hardcoded)

RESULTS_DIR = ROOT / "results"
CE_JSONL_PATH = RESULTS_DIR / f"wakevision_baseline-{BASELINE_NAME}_ce_results.jsonl"

all_results = _load_jsonl(CE_JSONL_PATH)

# De-duplicate (keep last record per key)
latest = {}
for r in all_results:
    if "model" in r and "corruption" in r:
        latest[(r["model"], r["corruption"])] = r

model_names = [name for name, _, _ in models]

# ---- Official mCE (15 common) + mCE-19 (all) ----
print("=" * 80)
print(f"WAKE VISION — mCE per model (baseline: {BASELINE_NAME})")
print("=" * 80)
print(f"{'Model':<20s} {'Clean Acc':>10s} {'mCE (%)':>10s} {'mCE-19 (%)':>12s}")
print("-" * 58)

for name in model_names:
    rows_all = [v for (m, c), v in latest.items() if m == name]
    rows_common = [v for (m, c), v in latest.items() if m == name and c in COMMON_CORRUPTIONS]
    if not rows_all:
        continue
    mce_15 = np.nanmean([r["ce_mean_1_5"] for r in rows_common]) * 100 if rows_common else float("nan")
    mce_19 = np.nanmean([r["ce_mean_1_5"] for r in rows_all]) * 100
    clean = clean_accs.get(name, float("nan"))
    obs = " <- baseline" if name == BASELINE_NAME else ""
    print(f"{name:<20s} {clean:>10.4f} {mce_15:>10.2f}% {mce_19:>12.2f}%{obs}")

print("-" * 58)
print("mCE    = 15 common corruptions (official metric, Hendrycks 2019)")
print("mCE-19 = all 19 corruptions (15 common + 4 extra/validation)")
print(f"\n{'Corruption':<24s} {'Type':<8s}", end="")
for name in model_names:
    print(f"{name:>16s}", end="")
print()
print("-" * (32 + 16 * len(model_names)))

for c in CORRUPTIONS:
    tag = "common" if c in COMMON_CORRUPTIONS else "extra"
    print(f"{c:<24s} {tag:<8s}", end="")
    for name in model_names:
        r = latest.get((name, c))
        if r:
            print(f"{r['ce_mean_1_5'] * 100:>16.2f}", end="")
        else:
            print(f"{'—':>16s}", end="")
    print()

WAKE VISION — mCE por model (baseline: MCUNet)
Model                Clean Acc    mCE (%)   mCE-19 (%)
----------------------------------------------------------
MobileNetV3Small         0.8613      71.91%        72.56%
EfficientNetB0           0.8773      58.72%        59.89%
MCUNet                   0.8127     100.00%       100.00% <- baseline
----------------------------------------------------------
mCE    = 15 common corruptions (official metric, Hendrycks 2019)
mCE-19 = todas as 19 corruptions (15 common + 4 extra/validation)

Corruption               Type   MobileNetV3Small  EfficientNetB0          MCUNet
--------------------------------------------------------------------------------
gaussian_noise           common             67.98           47.74          100.00
shot_noise               common             71.83           50.93          100.00
impulse_noise            common             70.17           47.25          100.00
defocus_blur             common             66.29     